# Phase 2.3 — XGBoost (Primary Model)

**Purpose:** Gradient-boosted trees as the primary model. Benchmark: LR Optimized AUC 0.7078, RF Tuned AUC 0.7571. XGBoost was predicted to reach 0.77–0.81; this notebook documents what actually happened and why.

**Key decisions inherited from prior phases:**
- Train/test split frozen since Phase 1.5 (300,429 / 75,108 rows)
- `scale_pos_weight = neg/pos ≈ 8.89` — XGBoost's native imbalance handling
- `is_reactivator` and `dormancy_days_before_reactivation` included (excluded from LR only)
- OrdinalEncoder for categoricals — trees don't need one-hot or scaling

**Last updated:** 2026-05-04

In [ ]:
import os, sys, json, time, warnings, logging
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from IPython.display import Image, display

from sklearn.preprocessing import OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    roc_curve, precision_recall_curve, auc as sk_auc,
    f1_score, precision_score, recall_score, roc_auc_score,
)
from sklearn.calibration import calibration_curve as sk_cal_curve
import xgboost as xgb
import optuna

logging.getLogger('optuna').setLevel(logging.WARNING)
warnings.filterwarnings('ignore')
os.environ['PYTHONIOENCODING'] = 'utf-8'

sys.path.insert(0, os.path.abspath('..'))
from src.config import (
    TRAIN_PATH, TEST_PATH, FIGURES, MODELS, TABLES,
    BENCHMARK_CSV, RANDOM_SEED
)
from src.evaluate import evaluate, print_metrics

FIG_DIR = FIGURES / '04_xgboost'
FIG_DIR.mkdir(parents=True, exist_ok=True)
print('Imports OK')

## 1. Load Data

Same frozen train/test as all prior phases. No re-splitting. `customers_master.parquet` was manually adjusted 2026-05-03 to introduce realistic recency-churn separation and behavioral heterogeneity — the adjustment that produced the LR-RF gap (+0.049) documented in Phase 2.2.

In [ ]:
train = pd.read_parquet(TRAIN_PATH)
test  = pd.read_parquet(TEST_PATH)

TARGET   = 'churned'
METADATA = ['wallet_id', 'full_name', 'churned_vendor']

FEATURE_COLS = [c for c in train.columns if c not in [TARGET] + METADATA]
CATEGORICAL_COLS = [
    'gender', 'state', 'city', 'referral_source',
    'preferred_language', 'linked_bank', 'kyc_tier',
]
NUMERIC_COLS = [c for c in FEATURE_COLS if c not in CATEGORICAL_COLS]

X_train = train[FEATURE_COLS]
y_train = train[TARGET].values
X_test  = test[FEATURE_COLS]
y_test  = test[TARGET].values

print(f'Train: {train.shape}  |  Test: {test.shape}')
print(f'Churn rate — train: {y_train.mean():.4f}  test: {y_test.mean():.4f}')
print(f'Features: {len(FEATURE_COLS)} total  ({len(NUMERIC_COLS)} numeric, {len(CATEGORICAL_COLS)} categorical)')

## 2. Preprocessing

Trees don't need scaling. `OrdinalEncoder` converts the 7 categorical columns to integer codes; `remainder='passthrough'` leaves numerics untouched. `handle_unknown='use_encoded_value'` with `unknown_value=-1` handles any unseen categories at inference time without crashing.

Fit on training set only — same discipline as every prior phase.

In [ ]:
ordinal_enc = OrdinalEncoder(
    handle_unknown='use_encoded_value',
    unknown_value=-1
)
preprocessor = ColumnTransformer(
    transformers=[('cat', ordinal_enc, CATEGORICAL_COLS)],
    remainder='passthrough',
    verbose_feature_names_out=False,
)
preprocessor.fit(X_train)
feat_names = preprocessor.get_feature_names_out()

X_tr_enc = preprocessor.transform(X_train)
X_te_enc = preprocessor.transform(X_test)

n_neg = int((y_train == 0).sum())
n_pos = int((y_train == 1).sum())
scale_pos_weight = n_neg / n_pos

print(f'Preprocessor fit OK. Output features: {len(feat_names)}')
print(f'scale_pos_weight = {n_neg}/{n_pos} = {scale_pos_weight:.4f}')

## 3. Vanilla XGBoost — Baseline

90/10 internal split from training data for early stopping. This split is carved from the training set — the frozen test set is never touched until final evaluation. `eval_metric='auc'` drives early stopping rather than log-loss, matching the primary evaluation metric.

In [ ]:
X_sub, X_val, y_sub, y_val = train_test_split(
    X_tr_enc, y_train,
    test_size=0.10,
    stratify=y_train,
    random_state=RANDOM_SEED
)

t0 = time.time()
vanilla_clf = xgb.XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=scale_pos_weight,
    eval_metric='auc',
    early_stopping_rounds=20,
    n_jobs=-1,
    random_state=RANDOM_SEED,
    tree_method='hist',
    verbosity=0,
)
vanilla_clf.fit(
    X_sub, y_sub,
    eval_set=[(X_val, y_val)],
    verbose=False
)
elapsed_vanilla = time.time() - t0
best_iter_vanilla = vanilla_clf.best_iteration
print(f'Vanilla done in {elapsed_vanilla:.1f}s | best_iteration={best_iter_vanilla}')

y_prob_vanilla = vanilla_clf.predict_proba(X_te_enc)[:, 1]
res_vanilla = evaluate(y_test, y_prob_vanilla)
print_metrics(res_vanilla, 'XGBoost_Vanilla')

In [ ]:
def safe_append(model_name, results, notes=''):
    row = {'model': model_name, **results, 'notes': notes}
    df_new = pd.DataFrame([row])
    if BENCHMARK_CSV.exists():
        df = pd.read_csv(BENCHMARK_CSV)
        if model_name in df['model'].values:
            print(f'[benchmark] {model_name} already present — skipping.')
            return
        df = pd.concat([df, df_new], ignore_index=True)
    else:
        df = df_new
    df.to_csv(BENCHMARK_CSV, index=False)
    print(f'[benchmark] appended {model_name}')

notes_vanilla = (
    f'n_estimators={best_iter_vanilla}, max_depth=6, lr=0.1, '
    f'scale_pos_weight={scale_pos_weight:.2f}, early_stopping=20'
)
safe_append('XGBoost_Vanilla', res_vanilla, notes=notes_vanilla)

**Vanilla result:** Early stopping fires at 35 trees — default lr=0.1 converges fast and shallow. Test AUC 0.759, PR-AUC 0.425. Already above RF Tuned (0.757 / 0.422) with no tuning. The gap over LR (+0.051 AUC) confirms that non-linear structure exists and XGBoost is exploiting it. PR-AUC 0.425 is the more informative metric here: at 10.1% base rate, precision-recall tradeoffs matter far more than the ROC diagonal.

## 4. Optuna Hyperparameter Search

**Search space:** n_estimators 300–1000, max_depth 4–10, learning_rate 0.01–0.3 (log scale), subsample 0.6–1.0, colsample_bytree 0.6–1.0, min_child_weight 1–10, gamma 0–5, reg_alpha 0–5, reg_lambda 0–5.

**Protocol:** 30 trials, TPESampler(seed=42), 5-fold stratified CV on the training set. Sequential trials (`n_jobs=1`) to avoid Windows fork overhead — XGBoost itself uses `n_jobs=-1` internally for tree-level parallelism. Total runtime: ~20 min.

The Optuna objective is defined below. If `xgb_results.json` exists (pre-computed by `run_xgb.py`), the study is skipped and best_params loaded directly — re-running the notebook stays fast. Delete the JSON to force a fresh search.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

def objective(trial):
    params = {
        'n_estimators':    trial.suggest_int('n_estimators', 300, 1000),
        'max_depth':       trial.suggest_int('max_depth', 4, 10),
        'learning_rate':   trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample':       trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree':trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_weight':trial.suggest_int('min_child_weight', 1, 10),
        'gamma':           trial.suggest_float('gamma', 0.0, 5.0),
        'reg_alpha':       trial.suggest_float('reg_alpha', 0.0, 5.0),
        'reg_lambda':      trial.suggest_float('reg_lambda', 0.0, 5.0),
        'scale_pos_weight':scale_pos_weight,
        'tree_method': 'hist', 'eval_metric': 'auc',
        'n_jobs': -1, 'random_state': RANDOM_SEED, 'verbosity': 0,
    }
    fold_aucs = []
    for tr_idx, va_idx in cv.split(X_tr_enc, y_train):
        clf = xgb.XGBClassifier(**params)
        clf.fit(X_tr_enc[tr_idx], y_train[tr_idx], verbose=False)
        proba = clf.predict_proba(X_tr_enc[va_idx])[:, 1]
        fold_aucs.append(roc_auc_score(y_train[va_idx], proba))
    return float(np.mean(fold_aucs))

results_path = TABLES / 'xgb_results.json'
if results_path.exists():
    with open(results_path, encoding='utf-8') as f:
        _res = json.load(f)
    best_params  = _res['best_params']
    best_cv_auc  = _res['cv_auc']
    n_trials_run = _res['n_trials_run']
    print(f'Loaded pre-computed Optuna results ({n_trials_run} trials, CV AUC={best_cv_auc:.4f})')
else:
    print('Running Optuna from scratch (~20 min)...')
    sampler = optuna.samplers.TPESampler(seed=RANDOM_SEED)
    study   = optuna.create_study(direction='maximize', sampler=sampler)
    study.optimize(objective, n_trials=30, n_jobs=1, show_progress_bar=True)
    best_params  = study.best_params
    best_cv_auc  = study.best_value
    n_trials_run = len(study.trials)

print(f'Best CV AUC: {best_cv_auc:.4f}')
print('Best params:')
for k, v in best_params.items():
    print(f'  {k}: {v}')

**Best trial (trial 24):** n_estimators=825, max_depth=6, lr=0.01025, subsample=0.705, colsample_bytree=0.830, min_child_weight=9, gamma=2.356, reg_alpha=2.611, reg_lambda=2.482.

Low learning rate (0.01) with 825 trees is the expected XGBoost sweet spot — more, smaller steps beat fewer, larger ones on tabular classification. Heavy regularization across all three penalties (alpha, lambda, gamma) confirms the data rewards shrinkage. `min_child_weight=9` prevents the tree from splitting on groups of fewer than 9 weighted samples, which directly controls overfitting on rare churners in a 10.1% minority class. The Bayesian optimizer converged on this combination after ~15 trials; the last 10 trials stayed in the same neighbourhood (CV AUC range 0.760–0.761).

## 5. Refit & Evaluate Tuned Model

Refit on the full training set using the Optuna best params, with the same 90/10 internal split for early stopping. The internal split fixes the number of trees (stops before overfitting); evaluation is then on the held-out test set.

In [ ]:
best_params_fit = {
    **best_params,
    'scale_pos_weight': scale_pos_weight,
    'eval_metric': 'auc',
    'early_stopping_rounds': 20,
    'n_jobs': -1,
    'random_state': RANDOM_SEED,
    'tree_method': 'hist',
    'verbosity': 0,
}

t0 = time.time()
tuned_clf = xgb.XGBClassifier(**best_params_fit)
tuned_clf.fit(X_sub, y_sub, eval_set=[(X_val, y_val)], verbose=False)
best_iter_tuned = tuned_clf.best_iteration
print(f'Refit done in {time.time()-t0:.1f}s | best_iteration={best_iter_tuned}')

y_prob_tuned = tuned_clf.predict_proba(X_te_enc)[:, 1]
res_tuned = evaluate(y_test, y_prob_tuned)
print_metrics(res_tuned, 'XGBoost_Tuned')

notes_tuned = (
    f'Optuna 30 trials, 5-fold CV; best_cv_auc={best_cv_auc:.4f}; '
    f'best_iter={best_iter_tuned}; '
    f"n_est={best_params.get('n_estimators')}, "
    f"depth={best_params.get('max_depth')}, "
    f"lr={best_params.get('learning_rate'):.4f}"
)
safe_append('XGBoost_Tuned', res_tuned, notes=notes_tuned)

**Tuned vs Vanilla:** Test AUC 0.7590 vs 0.7591 — difference is 0.0001, within measurement noise. The Optuna CV AUC (0.7614) doesn't fully transfer to the test set, which is expected: CV was estimated on the training distribution; the test set is genuinely held out. This is not a failure of tuning — it means the default XGBoost config was already near-optimal for this data. The tuned model uses 249 trees vs vanilla's 35, and shows marginally better PR-AUC (0.4258 vs 0.4255) and precision_at_k (0.3880 vs 0.3874). Going forward all analysis uses the tuned model as the primary; vanilla stays as the reference baseline.

**vs prior models:** XGBoost Tuned beats RF Tuned by +0.002 AUC and +0.004 PR-AUC. The predicted range was 0.77–0.81; actual is 0.759. The adjusted data's non-linear structure is richer than linear models can capture, but not rich enough to drive gradient boosting significantly above random forest. XGBoost's advantage shows more clearly in PR-AUC and calibration than in ROC-AUC.

## 6. Feature Importance — Three Methods

XGBoost exposes internal importance metrics (gain, weight/split-count) via the booster API. These are fast but biased toward high-cardinality features. Permutation importance on the test set corrects for this bias at the cost of compute (~5 repeats, ~2 min).

- **Gain:** Mean reduction in training loss per split using the feature
- **Weight:** Number of times the feature appears in any split across all trees
- **Permutation:** Mean AUC drop when the feature column is randomly shuffled (model-agnostic, reflects actual predictive contribution on test data)

Cross-referencing all three reduces the chance of acting on an artifact of one method.

In [ ]:
booster = tuned_clf.get_booster()
feat_idx_map = {f'f{i}': name for i, name in enumerate(feat_names)}

# Gain
gain_scores = booster.get_score(importance_type='gain')
gain_named = {feat_idx_map.get(k, k): v for k, v in gain_scores.items()}
gain_series = pd.Series(gain_named).sort_values(ascending=False)

# Weight
weight_scores = booster.get_score(importance_type='weight')
weight_named = {feat_idx_map.get(k, k): v for k, v in weight_scores.items()}
weight_series = pd.Series(weight_named).sort_values(ascending=False)

# Permutation
perm_result = permutation_importance(
    tuned_clf, X_te_enc, y_test,
    n_repeats=5, scoring='roc_auc', n_jobs=-1, random_state=RANDOM_SEED
)
perm_series = pd.Series(perm_result.importances_mean, index=feat_names).sort_values(ascending=False)

# Save comparison CSV
TOP_N = 20
top20 = gain_series.head(TOP_N)
gain_rank_map   = {f: i+1 for i, f in enumerate(gain_series.index)}
weight_rank_map = {f: i+1 for i, f in enumerate(weight_series.index)}
perm_rank_map   = {f: i+1 for i, f in enumerate(perm_series.index)}

imp_df = pd.DataFrame({
    'feature':     top20.index,
    'gain_score':  top20.values,
    'gain_rank':   [gain_rank_map[f] for f in top20.index],
    'weight_rank': [weight_rank_map.get(f, 999) for f in top20.index],
    'perm_rank':   [perm_rank_map.get(f, 999) for f in top20.index],
    'perm_score':  [perm_series.get(f, 0.0) for f in top20.index],
})
imp_df.to_csv(TABLES / 'xgb_feature_importance.csv', index=False)

print('Top 10 by Gain:')
print(gain_series.head(10).to_string())
print('\nTop 10 by Permutation:')
print(perm_series.head(10).to_string())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 7))

gain_series.head(20).sort_values().plot(kind='barh', ax=axes[0], color='#2196F3')
axes[0].set_title('Gain (top 20)', fontsize=11)
axes[0].set_xlabel('Mean Gain per Split')

weight_series.head(20).sort_values().plot(kind='barh', ax=axes[1], color='#FF9800')
axes[1].set_title('Weight/Count (top 20)', fontsize=11)
axes[1].set_xlabel('Split Count')

perm_series.head(20).sort_values().plot(kind='barh', ax=axes[2], color='#4CAF50')
axes[2].set_title('Permutation AUC Drop (top 20)', fontsize=11)
axes[2].set_xlabel('Mean AUC Drop')

plt.suptitle('XGBoost Feature Importance — Three Methods', fontsize=13)
plt.tight_layout()
plt.savefig(FIG_DIR / 'gain_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved.')

**Gain top features:** `dormancy_days_before_reactivation` (8,237) and `is_reactivator` (6,392) dominate — these are the reactivator-segment features excluded from LR. This is exactly what the Phase 2.2 hypothesis predicted: the non-linear structure introduced by the data adjustment is concentrated in the reactivator segment, which trees can split on but linear models cannot.

`days_since_last_txn` ranks 3rd by gain (864) and **1st by permutation** (AUC drop 0.074). The permutation ranking is the more reliable signal: it measures actual contribution to held-out test AUC, not just training-time split quality. This confirms the EDA finding (Spearman ρ=0.15, rank 2) and the LR finding (coefficient 0.518, rank 2 of 55).

Gain and permutation rankings diverge for the reactivator features — gain ranks them 1–2 because the dormancy split creates very pure nodes (high gain), but permutation ranks `is_reactivator` only 3rd (AUC drop 0.021). Both readings are valid: the dormancy/reactivation structure creates high-confidence predictions for that specific sub-population, but the broader population's AUC is driven more by `days_since_last_txn` and `txn_velocity_change`.

## 7. 4-Model ROC/PR Comparison

Four models evaluated on the same frozen test set: LR Optimized, RF Tuned, XGBoost Vanilla, XGBoost Tuned. AUC is the primary ranking metric; PR-AUC captures precision-recall tradeoff at the actual 10.1% base rate.

In [ ]:
lr_pipe = joblib.load(MODELS / 'logistic.pkl')
rf_pipe = joblib.load(MODELS / 'random_forest.pkl')

lr_feat_in = lr_pipe.named_steps['preprocessor'].feature_names_in_
rf_feat_in = rf_pipe.named_steps['preprocessor'].feature_names_in_

y_prob_lr = lr_pipe.predict_proba(test[list(lr_feat_in)])[:, 1]
y_prob_rf = rf_pipe.predict_proba(test[list(rf_feat_in)])[:, 1]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
models_plot = {
    'LR_Optimized': (y_prob_lr,      '#4C72B0'),
    'RF_Tuned':      (y_prob_rf,      '#DD8452'),
    'XGB_Vanilla':   (y_prob_vanilla, '#55A868'),
    'XGB_Tuned':     (y_prob_tuned,   '#C44E52'),
}
for name, (proba, color) in models_plot.items():
    fpr, tpr, _ = roc_curve(y_test, proba)
    ax1.plot(fpr, tpr, label=f'{name} ({sk_auc(fpr,tpr):.4f})', color=color, lw=2)
    prec_c, rec_c, _ = precision_recall_curve(y_test, proba)
    ax2.plot(rec_c, prec_c, label=f'{name} ({sk_auc(rec_c,prec_c):.4f})', color=color, lw=2)

ax1.plot([0,1],[0,1],'k--',lw=1); ax1.set_xlabel('FPR'); ax1.set_ylabel('TPR')
ax1.set_title('ROC — 4-Model Comparison'); ax1.legend(loc='lower right', fontsize=8)

ax2.axhline(y_test.mean(), color='k', ls='--', lw=1, label=f'Baseline ({y_test.mean():.3f})')
ax2.set_xlabel('Recall'); ax2.set_ylabel('Precision')
ax2.set_title('PR — 4-Model Comparison'); ax2.legend(loc='upper right', fontsize=8)

plt.suptitle('Phase 2.3 — XGBoost vs Prior Models', fontsize=13)
plt.tight_layout()
plt.savefig(FIG_DIR / 'roc_pr_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

**ROC comparison:** The AUC ordering is XGB_Vanilla (0.759) ≈ XGB_Tuned (0.759) > RF_Tuned (0.757) >> LR (0.708). The XGBoost models separate from RF by ~0.002 — real but small. The LR-to-tree gap (+0.051) is larger and confirms that non-linear structure exists in the data.

**PR comparison:** At 10.1% churn, the PR curve is more informative. XGBoost Tuned PR-AUC 0.426 vs RF 0.422 — marginal improvement, but XGBoost concentrates its precision gains at lower recall values. Both tree models massively outperform LR on PR-AUC (0.257), which makes sense: LR can't capture the segment-conditional structure that drives precision at high threshold.

## 8. Calibration

Well-calibrated probabilities matter for retention interventions: if the model says 0.6 churn risk, the business expects ~60% of those customers to actually churn. MACE (Mean Absolute Calibration Error) measures average deviation between predicted probability bins and observed fraction of positives.

XGBoost with `scale_pos_weight` inflates raw scores by the class weight ratio. The calibration curve shows whether this inflation is systematic or variable.

In [ ]:
frac_xgb, mean_xgb = sk_cal_curve(y_test, y_prob_tuned, n_bins=10, strategy='uniform')
frac_lr,  mean_lr  = sk_cal_curve(y_test, y_prob_lr, n_bins=10, strategy='uniform')
frac_rf,  mean_rf  = sk_cal_curve(y_test, y_prob_rf, n_bins=10, strategy='uniform')

mace_xgb = float(np.mean(np.abs(frac_xgb - mean_xgb)))
mace_lr  = float(np.mean(np.abs(frac_lr  - mean_lr)))
mace_rf  = float(np.mean(np.abs(frac_rf  - mean_rf)))
print(f'MACE — LR: {mace_lr:.4f}  RF: {mace_rf:.4f}  XGB_Tuned: {mace_xgb:.4f}')

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot([0,1],[0,1],'k--',lw=1,label='Perfect calibration')
ax.plot(mean_lr,  frac_lr,  'o-', label=f'LR (MACE={mace_lr:.3f})',      color='#4C72B0')
ax.plot(mean_rf,  frac_rf,  's-', label=f'RF_Tuned (MACE={mace_rf:.3f})', color='#DD8452')
ax.plot(mean_xgb, frac_xgb, '^-', label=f'XGB_Tuned (MACE={mace_xgb:.3f})',
        color='#C44E52', lw=2, ms=8)
ax.set_xlabel('Mean predicted probability'); ax.set_ylabel('Fraction of positives')
ax.set_title('Calibration Curve — LR vs RF vs XGB')
ax.legend(fontsize=9); ax.set_xlim(0,1); ax.set_ylim(0,1)
plt.tight_layout()
plt.savefig(FIG_DIR / 'calibration_curve.png', dpi=150, bbox_inches='tight')
plt.show()

**Calibration results:** MACE — LR: 0.3260, RF: 0.2758, XGB_Tuned: 0.3025.

All three models are poorly calibrated in absolute terms; all require isotonic regression or Platt scaling before deployment. The structural cause is `scale_pos_weight`/`class_weight`: inflating minority-class signal shifts predicted probabilities upward from the 10.1% base rate.

RF calibrates slightly better than XGBoost (0.276 vs 0.303), which is a known pattern — random forests produce more moderate probability estimates through averaging, while boosting pushes scores toward the extremes. XGBoost is significantly better than LR (0.303 vs 0.326). All three models need post-hoc calibration before their output probabilities can be used directly as risk scores in a retention intervention system — a calibration layer is included in Phase 2.6 (imbalance study).

## 9. Threshold Analysis

Binary classification defaults to threshold=0.5, but this is almost never optimal for imbalanced data. Two operationally meaningful thresholds:
- **F1-optimal:** maximizes F1 score — balanced precision/recall
- **Recall≥0.80:** highest precision while catching ≥80% of churners (retention-first strategy)

At 10.1% base rate with `scale_pos_weight=8.89`, the model outputs inflated scores. F1-optimal threshold is expected to be well above 0.5.

In [ ]:
thresholds = np.arange(0.10, 0.95, 0.05)
precisions_t, recalls_t, f1s_t = [], [], []
for thr in thresholds:
    yp = (y_prob_tuned >= thr).astype(int)
    precisions_t.append(precision_score(y_test, yp, zero_division=0))
    recalls_t.append(recall_score(y_test, yp, zero_division=0))
    f1s_t.append(f1_score(y_test, yp, zero_division=0))

precisions_t = np.array(precisions_t)
recalls_t    = np.array(recalls_t)
f1s_t        = np.array(f1s_t)

best_f1_idx = int(np.argmax(f1s_t))
best_f1_thr = float(thresholds[best_f1_idx])
rc_mask     = recalls_t >= 0.80
rc_thr = float(thresholds[rc_mask][np.argmax(precisions_t[rc_mask])]) if rc_mask.any() else None

print(f'F1-optimal threshold:  {best_f1_thr:.2f}')
print(f'F1 at optimal: {f1s_t[best_f1_idx]:.4f}')
if rc_thr:
    rc_idx = list(thresholds).index(rc_thr)
    print(f'Recall>=0.80 threshold: {rc_thr:.2f}  precision={precisions_t[rc_idx]:.4f}  recall={recalls_t[rc_idx]:.4f}')

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(thresholds, precisions_t, label='Precision', color='#4C72B0', lw=2)
ax.plot(thresholds, recalls_t,    label='Recall',    color='#DD8452', lw=2)
ax.plot(thresholds, f1s_t,        label='F1 Score',  color='#55A868', lw=2)
ax.axvline(0.5,         color='gray',    ls='--', lw=1.2, label='Default (0.5)')
ax.axvline(best_f1_thr, color='#55A868', ls=':',  lw=1.5, label=f'F1-optimal ({best_f1_thr:.2f})')
if rc_thr:
    ax.axvline(rc_thr, color='#DD8452', ls=':', lw=1.5, label=f'Recall>=0.8 ({rc_thr:.2f})')
ax.set_xlabel('Decision Threshold'); ax.set_ylabel('Score')
ax.set_title('Threshold Analysis — XGBoost_Tuned')
ax.set_xlim(0.10, 0.90); ax.set_ylim(0, 1)
ax.legend(loc='upper right', fontsize=9)
plt.tight_layout()
plt.savefig(FIG_DIR / 'threshold_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

**Threshold findings:** F1-optimal threshold is 0.65, recall≥0.80 threshold is 0.35.

The high F1-optimal (0.65) is explained by `scale_pos_weight=8.89`: the model pushes churner probabilities high and retained-customer probabilities low, so the natural separation point is well above 0.5. Deploying at the default 0.5 threshold captures most churners (recall 0.61) but at low precision (0.23). At 0.65, precision improves significantly at the cost of recall.

For a retention campaign, the Recall≥0.80 threshold (0.35) is more operationally relevant: catching 80% of churners is the typical goal, and 0.35 achieves this. The tradeoff is more false positives — sending retention offers to customers who would have stayed. The cost-of-false-positive is low (wasted offer cost) relative to cost-of-false-negative (lost customer), which justifies the lower threshold for business deployment.

In [ ]:
tuned_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', tuned_clf),
])
joblib.dump(tuned_pipe, MODELS / 'xgboost.pkl')

# Smoke test
loaded = joblib.load(MODELS / 'xgboost.pkl')
smoke = loaded.named_steps['classifier'].predict_proba(
    loaded.named_steps['preprocessor'].transform(X_test.head(5))
)[:, 1]
print(f'Saved: {MODELS / "xgboost.pkl"}')
print(f'Smoke test (5 rows): {smoke.round(4).tolist()}')

## 10. Benchmark Summary & Phase 2.3 Findings

Final benchmark after Phase 2.3:

In [ ]:
bench = pd.read_csv(BENCHMARK_CSV)
display(bench[['model','auc','pr_auc','f1','precision','recall','precision_at_k','recall_at_k']].round(4))

print(f'\nAUC delta XGB_Tuned vs RF_Tuned: {res_tuned["auc"] - 0.757060:+.4f}')
print(f'AUC delta XGB_Tuned vs LR_Opt:   {res_tuned["auc"] - 0.707783:+.4f}')

### Phase 2.3 Findings

1. **XGBoost marginally beats RF (+0.002 AUC, +0.004 PR-AUC).** Both tree models are in the same performance band. The predicted 0.77–0.81 ceiling was not reached; the adjusted data supports non-linear structure but not deep boosting gains over random forest.

2. **Tuning adds nothing on AUC but improves regularization.** Optuna's best config (30 trials, CV AUC 0.7614) produces test AUC 0.7590 — statistically identical to vanilla (0.7591). The value of tuning is documented regularization choices, not metric improvement.

3. **Reactivator features are the dominant gain signal.** `dormancy_days_before_reactivation` and `is_reactivator` rank 1–2 by gain. These were excluded from LR and are the main mechanism behind the LR-to-tree gap (+0.051 AUC). Permutation importance correctly downgrades them — their contribution is concentrated in a specific sub-population (5,757 wallets), not the full test set.

4. **Recency (`days_since_last_txn`) is the universal predictor.** Rank 2 LR coefficient, rank 3 XGB gain, rank 1 XGB permutation. Consistent across all three modeling phases.

5. **All models need calibration before deployment.** MACE: LR 0.326, XGB 0.303, RF 0.276. The Phase 2.5 imbalance study will include isotonic calibration as a treatment arm.

6. **Recommended deployment threshold: 0.35 (Recall≥0.80 strategy).** F1-optimal is 0.65 but under-serves the retention-campaign use case. The imbalance study (Phase 2.5) will formalize cost-sensitive threshold selection.

**Next:** Phase 2.4 — LightGBM + CatBoost head-to-head with XGBoost.